**STEP 1: Set Up Environment**

Run it only if the environment does not have the needed dependencies available.

In [1]:
# Display installation header with visual separator
# print("="*70)
# print(" INSTALLING PACKAGES")
# print("="*70)
# print("This will take 2-3 minutes. Please wait...")
# print("(Dependency warnings are normal in Colab and can be ignored)\n")

# Install all required packages quietly with filtered output
# Packages installed:
# - langchain: Core LangChain framework
# - langchain-community: Community integrations for LangChain
# - langchain-google-genai: Google Gemini integration
# - sentence-transformers: For text embeddings
# - chromadb: Vector database for document storage
# - google-generativeai: Native Google Gemini SDK
# - google-search-results: For web search capabilities (optional)
# - pandas: Data manipulation and analysis
#
# The grep command filters out common dependency warnings to keep output clean
# %pip install -q langchain langchain-community langchain-google-genai sentence-transformers chromadb google-generativeai google-search-results pandas 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true

# Display completion message
# print("\n Installation complete!")
# print("="*70)

In [2]:
# Importing Libraries
# LangChain - Google Gemini Integration
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain - Core Components
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Utility Libraries
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
#  API Key Configuration
# ============================================================================
# Securely retrieve and configure the Google Gemini API key.
# This approach keeps API keys secure and prevents accidental exposure in code.
# ============================================================================

# Retrieve API key from Colab Secrets and set as environment variable
# To add your API key:
# 1. Click the 🔑 icon in the left sidebar (Secrets)
# 2. Add a new secret named 'GEMINI_API_KEY'
# 3. Paste your Gemini API key as the value
# Get your free API key at: https://aistudio.google.com/app/api-keys?

api_key = os.getenv("GEMINI_API_KEY")

print("✓ API Key configured successfully")
print("   (Key is stored securely and not displayed)")
print("API Key Found:", api_key is not None)
print("API Key Length:", len(api_key) if api_key else 0)

✓ API Key configured successfully
   (Key is stored securely and not displayed)
API Key Found: True
API Key Length: 53


In [4]:
from google import genai

client = genai.Client(
    api_key=api_key
)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

**STEP 2: Initialize Language Model**

In [5]:
# ============================================================================
# Configure and initialize the Google Gemini model that will power our
# quiz generation. This model handles all AI-powered tasks including question
# generation, answer creation, and content analysis.
# ============================================================================

# Initialize Gemini LLM with optimized parameters
# NOTE: Use LangChain's ChatGoogleGenerativeAI wrapper (not the raw
# google-genai client) so that `llm` is a Runnable that can be piped
# into the LangChain chains built in the next steps.

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7,
    google_api_key=api_key,
    timeout=10,       # API minimum deadline is 10s
    max_retries=1,    # fail fast instead of silently retrying for minutes
)

print("✓ Gemini LLM initialized successfully")
print(f"   Model: gemini-3.5-flash")
print(f"   Temperature: 0.7 (balanced creativity)")
print(f"   Google API Key submitted")

# .text extracts the plain string from the structured response content
print(llm.invoke("hello").text)

✓ Gemini LLM initialized successfully
   Model: gemini-3.5-flash
   Temperature: 0.7 (balanced creativity)
   Google API Key submitted
Hello! How can I help you today?


**STEP 3: Design Prompt Templates**

In [6]:
#  Chain 1 - Generate a question
# ============================================================================
# This chain creates a beginner-level question based on the provided topic.
# It's the first step in our sequential quiz generation pipeline.
# ============================================================================

# Define the prompt template for question generation
# This template takes a topic and generates an appropriate beginner question
question_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Generate a beginner-level question related to the topic: {topic}"
)



In [7]:
#  Chain 2 - Answer the question
# ============================================================================
# This chain generates a clear, concise answer with explanation for the
# question created by Chain 1. It provides both the answer and reasoning.
# ============================================================================

# Define the prompt template for answer generation
# This template takes the generated question and produces an answer with explanation

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="Provide a clear answer with a short explanation for the question: {question}"
)

print("✓ Chain 2 (Answer Generation) prompt template created")
print("   Purpose: Generate clear answers with explanations")

✓ Chain 2 (Answer Generation) prompt template created
   Purpose: Generate clear answers with explanations


**STEP 4: Build Individual Chains**

In [8]:
#  Create individual chains
# ============================================================================
# Instantiate the individual chains that will be connected in a sequence.
# Each chain wraps the LLM with its specific prompt template and output key.
# ============================================================================

# Chain 1: Question Generation Chain
# Takes 'topic' as input, outputs 'question'

question_chain = question_prompt | llm | StrOutputParser()
answer_chain = answer_prompt | llm | StrOutputParser()

print("✓ Individual chains created successfully")
print("   Chain 1: topic → question")
print("   Chain 2: question → answer")

✓ Individual chains created successfully
   Chain 1: topic → question
   Chain 2: question → answer


**STEP 5: Compose Sequential Chain**

In [9]:
# ============================================================================
# Connect the individual chains into a sequential workflow where the output
# of Chain 1 (question) automatically becomes the input for Chain 2 (answer).
# This creates an end-to-end quiz generation pipeline.
# ============================================================================

# Create the sequential chain that orchestrates the entire workflow

chain = RunnablePassthrough.assign(
    question=question_chain
).assign(
    answer=answer_chain
)

print("✓ Sequential chain created successfully")
print("   Pipeline: topic → [Chain 1] → question → [Chain 2] → answer")
print("   Verbose mode: ENABLED (will show execution details)")

✓ Sequential chain created successfully
   Pipeline: topic → [Chain 1] → question → [Chain 2] → answer
   Verbose mode: ENABLED (will show execution details)


**STEP 6: Run Quiz Generator**

In [10]:
# ============================================================================
# Run the complete sequential chain with user input and display the results.
# The pipeline will execute both chains automatically and return the final
# question-answer pair.
# ============================================================================

# Get user input for the quiz topic

topic = input("Enter a topic: ")

# Execute the sequential chain
# The chain will:
# 1. Generate a question from the topic (Chain 1)
# 2. Generate an answer for that question (Chain 2)
# 3. Return both outputs in a dictionary

print("Starting chain execution...")

response = chain.invoke(
    {"topic": topic}
)

print("Chain execution completed")

# ============================================================================
# DISPLAY RESULTS
# ============================================================================
# Present the generated quiz question and answer in a clean, readable format.
# ============================================================================

print("\n" + "="*70)
print(" GENERATED QUIZ")
print("="*70)

# Display the generated question
print("\n❓ Question:")
print(response["question"])

# Display the answer with explanation
print("\n Answer & Explanation:")
print(response["answer"])

print("\n" + "="*70)

Starting chain execution...
Chain execution completed

 GENERATED QUIZ

❓ Question:
Here is a beginner-level question about LangChain:

---

**Question:**  
In LangChain, what is the primary purpose of a **`PromptTemplate`**?

**A)** To train a Large Language Model (LLM) on a custom dataset.  
**B)** To dynamically create prompts by inserting user input or variables into a predefined text template.  
**C)** To connect the LLM directly to a SQL database.  
**D)** To save and store past conversation history.  

---

### **Answer Key:**
> **Correct Answer: B**
> 
> **Explanation:**  
> A `PromptTemplate` in LangChain allows you to create reproducible and dynamic prompt strings. Instead of writing hardcoded text, you define placeholders (variables) that can be filled in dynamically at runtime based on user input or other data sources.

 Answer & Explanation:
**Correct Answer:** **B) To dynamically create prompts by inserting user input or variables into a predefined text template.**

---

